# Population Preprocessing Notebook

In [3]:
import pandas as pd
import os

Reading the population data into a dataframe

In [16]:
# Assumes python is running from this notebook
# If your current directory is anything else change this to the relative path of the
# policeforceareas1991to2024.xlsx file
population_path = f"{os.path.dirname(os.path.normpath(os.getcwd()))}/data/raw/policeforceareas1991to2024.xlsx"


In [31]:
# Assumes you want to save the output file in the processed folder.
# If not, change this to the relative path of the folder you wish to export the processed file to
processed_folder = f"{os.path.dirname(os.path.normpath(os.getcwd()))}/data/processed"

In [17]:
# Reading Population Data into a dataframe
raw_population_data = pd.read_excel(
    population_path,
    sheet_name='Mid-2021 to Mid-2024',
    skiprows=3
)

In [24]:
# Filtering Forces and Years of Interest
forces = [
    'Metropolitan Police',
    'Devon & Cornwall',
    'North Yorkshire',
    'South Wales',
    'Thames Valley'
]

pop_df = raw_population_data[
    raw_population_data['PFA 2023 Name'].isin(forces)
    & raw_population_data['Year'].isin([2023, 2024])
].copy()

In [25]:
pop_df.shape
pop_df.info()
pop_df

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 2 to 167
Columns: 175 entries, PFA 2023 Code to M85
dtypes: int64(173), object(2)
memory usage: 13.8+ KB


,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M76,M77,M78,M79,M80,M81,M82,M83,M84,M85
2,E23000001,Metropolitan Police,2023,51047,53405,51051,51359,50792,50682,52018,...,23619,18192,16512,15945,14302,12562,10881,11329,10369,53824
3,E23000001,Metropolitan Police,2024,51526,51041,52918,50777,51276,50664,50846,...,21814,22697,17361,15658,15144,13481,11793,10164,10433,55460
34,E23000009,North Yorkshire,2023,3137,3422,3464,3793,3716,3846,3901,...,5392,3923,3773,3636,3143,2668,2165,2258,2109,10407
35,E23000009,North Yorkshire,2024,3028,3248,3582,3581,3877,3827,3909,...,4885,5211,3800,3620,3484,2977,2547,2033,2085,10803
114,E23000029,Thames Valley,2023,13189,13995,13976,14478,14920,15175,15813,...,10798,7861,7677,7555,6546,5807,4839,4907,4523,23997
115,E23000029,Thames Valley,2024,12969,13731,14511,14554,15016,15413,15555,...,10026,10483,7619,7412,7226,6252,5498,4499,4561,24932
138,E23000035,Devon & Cornwall,2023,6871,7322,7644,8001,8379,8480,8634,...,12369,9348,8777,8333,7480,6119,5148,5014,4643,23640
139,E23000035,Devon & Cornwall,2024,6714,7031,7527,7821,8193,8577,8649,...,10898,12006,9009,8384,7986,7100,5789,4832,4625,24273
166,W15000003,South Wales,2023,5893,6291,6261,6773,6917,7109,7262,...,6396,4737,4601,4351,3748,3360,2920,2721,2429,12159
167,W15000003,South Wales,2024,5834,6011,6383,6338,6919,7015,7244,...,5996,6138,4506,4402,4109,3501,3143,2706,2512,12387


### Observations

- Population is spread across age columns F0–F85 (female) and M0–M85 (male); the sum of these should be an estimate of the population in each police force
- Mid-year estimates are labelled by year (2021–2024), so we will use mid-2023 and mid-2024 as our Year 1 and Year 2 estimates respectively

##### Combining the Population Estimates to a single value

In [26]:
# Ignore none estimate columns
age_columns = [col for col in pop_df.columns if col.startswith('F') or col.startswith('M')]

pop_df['Total Population'] = pop_df[age_columns].sum(axis=1)

In [27]:
pop_df = pop_df[['PFA 2023 Name', 'Year', 'Total Population']]
pop_df = pop_df.rename(columns={
    'PFA 2023 Name': 'force',
    'Year': 'mid_year'
})

In [28]:
pop_df['analysis_year'] = pop_df['mid_year'].map({2023: 'Year 1', 2024: 'Year 2'})

In [29]:
# Data Quality Checks
pop_df.shape
pop_df.info()
pop_df

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 2 to 167
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   force             10 non-null     object
 1   mid_year          10 non-null     int64 
 2   Total Population  10 non-null     int64 
 3   analysis_year     10 non-null     object
dtypes: int64(2), object(2)
memory usage: 400.0+ bytes


,force,mid_year,Total Population,analysis_year
2,Metropolitan Police,2023,8986099,Year 1
3,Metropolitan Police,2024,9074625,Year 2
34,North Yorkshire,2023,835592,Year 1
35,North Yorkshire,2024,844571,Year 2
114,Thames Valley,2023,2600732,Year 1
115,Thames Valley,2024,2640201,Year 2
138,Devon & Cornwall,2023,1825690,Year 1
139,Devon & Cornwall,2024,1840161,Year 2
166,South Wales,2023,1353346,Year 1
167,South Wales,2024,1363561,Year 2


In [33]:
pop_df.to_csv(f"{processed_folder}/population_master.csv", index=False)